<h1>Chapter 11 - Fine-tuning Representation Models for Classification</h1>
<i>Exploring the performance in classification of representation models.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter11/Chapter%2011%20-%20Fine-Tuning%20BERT.ipynb)

---

This notebook is for Chapter 11 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
%%capture
!pip install -U "datasets>=2.18.0" transformers accelerate sentence-transformers setfit seqeval evaluate

## **Data**


> **📖 O que está acontecendo aqui?**
>
> Carregamos o dataset **Rotten Tomatoes**, composto por críticas de filmes rotuladas como positivas ou negativas.
> São 5.331 reviews positivos e 5.331 negativos. O `load_dataset` do HuggingFace já divide automaticamente em splits `train` e `test`.
> Esse dataset será utilizado como tarefa de **classificação de sentimento** ao longo de todo o capítulo.


In [ ]:
from datasets import load_dataset

# Prepare data and splits
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

## **Supervised Classification**


> **📖 O que é Supervised Classification aqui?**
>
> Nesta seção fazemos **fine-tuning completo** de um BERT pré-treinado para classificação de sentimento.
> Diferente de usar o modelo congelado (como no Capítulo 4), aqui **todos os pesos são atualizados** durante o treino.
> O processo segue 4 etapas: carregar modelo → tokenizar dados → definir métricas → treinar e avaliar.


### HuggingFace Trainer


> **📖 O que está acontecendo aqui?**
>
> `AutoModelForSequenceClassification` carrega o BERT e adiciona automaticamente uma **classification head** (camada linear) para `num_labels=2` classes.
> `AutoTokenizer` carrega o tokenizador correspondente, que converte texto em IDs numéricos.
> O modelo base `bert-base-cased` foi pré-treinado na Wikipedia e em livros — ele é sensível a maiúsculas/minúsculas.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load Model and Tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenize our data.


> **📖 O que está acontecendo aqui?**
>
> O BERT não lê texto diretamente — precisa de **IDs numéricos**. O tokenizador converte cada texto em tokens e depois em IDs.
> `truncation=True` garante que textos muito longos sejam cortados no limite de 512 tokens do modelo.
> `DataCollatorWithPadding` adiciona tokens `[PAD]` para igualar o tamanho das sequências dentro de cada batch.
> `batched=True` processa múltiplos exemplos ao mesmo tempo, tornando a tokenização muito mais rápida.
>
> 💡 Exemplo: `"What a horrible movie!"` → `[CLS] What a horrible movie ! [SEP]` → `[101, 1327, 170, 17551, 2523, 106, 102]`


In [ ]:
from transformers import DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)


Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Define metrics.


> **📖 O que está acontecendo aqui?**
>
> Definimos a métrica de avaliação: o **F1 Score**, que varia de 0 a 1 (quanto mais próximo de 1, melhor).
> `np.argmax(logits, axis=-1)` converte as saídas brutas do modelo (logits) na classe prevista (0 ou 1).
> Essa função é chamada automaticamente pelo `Trainer` ao final de cada avaliação.


In [ ]:
import numpy as np
import evaluate


def compute_metrics(eval_pred):
    """Calculate F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    load_f1 = evaluate.load("f1")
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}


Train model.


> **📖 O que está acontecendo aqui?**
>
> `TrainingArguments` define os hiperparâmetros do treino:
> - `learning_rate=2e-5` → taxa de aprendizado pequena, padrão para fine-tuning de transformers
> - `per_device_train_batch_size=16` → 16 exemplos processados por passo
> - `num_train_epochs=1` → uma passagem completa pelo dataset de treino
> - `weight_decay=0.01` → regularização L2 para reduzir overfitting
>
> O `Trainer` executa o loop completo: **forward pass → loss → backpropagation → atualização de pesos**.
> Neste cenário, **todos os parâmetros do BERT + classification head são atualizados**.


In [ ]:
from transformers import TrainingArguments, Trainer

# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none",
   disable_tqdm=True
)

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()


{'loss': '0.4017', 'grad_norm': '8.289', 'learning_rate': '1.311e-06', 'epoch': '0.9363'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '118', 'train_samples_per_second': '72.32', 'train_steps_per_second': '4.527', 'train_loss': '0.3986', 'epoch': '1'}


TrainOutput(global_step=534, training_loss=0.3986303690220979, metrics={'train_runtime': 117.9502, 'train_samples_per_second': 72.319, 'train_steps_per_second': 4.527, 'train_loss': 0.3986303690220979, 'epoch': 1.0})

Evaluate results.


> **📖 O que está acontecendo aqui?**
>
> `trainer.evaluate()` avalia o modelo no conjunto de **teste** (dados nunca vistos durante o treino).
> Retorna `eval_loss` (perda), `eval_f1` (nosso F1 Score — esperado **~0.85**) e métricas de tempo de execução.


In [ ]:
trainer.evaluate()


{'eval_loss': '0.358', 'eval_f1': '0.8523', 'eval_runtime': '3.994', 'eval_samples_per_second': '266.9', 'eval_steps_per_second': '16.78', 'epoch': '1'}


{'eval_loss': 0.3579951822757721,
 'eval_f1': 0.8523364485981308,
 'eval_runtime': 3.9936,
 'eval_samples_per_second': 266.926,
 'eval_steps_per_second': 16.777,
 'epoch': 1.0}

---
## 📝 Questão 1 — Supervised Classification

**Verdadeiro ou Falso — marque (V) ou (F) e justifique as falsas:**

( ) 1. O modelo `bert-base-cased` distingue letras maiúsculas de minúsculas durante a tokenização.

( ) 2. O parâmetro `truncation=True` no tokenizador faz com que textos mais curtos que 512 tokens sejam descartados.

( ) 3. O `DataCollatorWithPadding` garante que todas as sequências de um mesmo batch tenham o mesmo tamanho.

( ) 4. O F1 Score é a métrica utilizada para avaliar o modelo neste notebook.

( ) 5. Com `num_train_epochs=1`, o modelo passa pelos dados de treino duas vezes.

( ) 6. `weight_decay=0.01` é uma técnica de regularização que ajuda a evitar overfitting.


**Questão dissertativa:**

Explique com suas próprias palavras por que o fine-tuning completo (treinar todos os parâmetros) tende a resultar em um F1 Score maior do que usar o modelo pré-treinado congelado. O que muda durante o treino?


### Freeze Layers


> **📖 O que é Freezing de Camadas?**
>
> Congelar uma camada significa impedir que seus pesos sejam atualizados durante o backpropagation.
> Isso é feito setando `requires_grad = False` nos parâmetros desejados — o PyTorch simplesmente não calcula gradientes para eles.
>
> 🎸 **Analogia:** Um músico clássico que quer aprender rock não precisa reaprender teoria musical — só treina as partes novas.
> As camadas congeladas são o "conhecimento base" que já está pronto; apenas o topo (a classification head) aprende a nova tarefa.
>
> Aqui reiniciamos o modelo para testar estratégias de congelamento do zero.


In [ ]:
# Load Model and Tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


> **📖 O que está acontecendo aqui?**
>
> `model.named_parameters()` retorna o nome e os pesos de cada parâmetro do modelo.
> Inspecionando esses nomes vemos a estrutura interna do BERT:
> - `bert.embeddings.*` → embeddings de palavras, posições e tipos de tokens
> - `bert.encoder.layer.0.*` até `bert.encoder.layer.11.*` → 12 Encoder Blocks (atenção + feedforward)
> - `bert.pooler.*` → camada que agrega as representações
> - `classifier.*` → a classification head que foi adicionada automaticamente


In [ ]:
# Print layer names
for name, param in model.named_parameters():
    print(name)


bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

> **📖 O que está acontecendo aqui?**
>
> Aplicamos o **freeze total**: todos os parâmetros recebem `requires_grad = False`, exceto os que começam com `"classifier"`.
> Isso significa que apenas a classification head (duas camadas lineares) será atualizada durante o treino.
> Todo o BERT — embeddings, 12 encoders, pooler — fica completamente estático.


In [ ]:
for name, param in model.named_parameters():

     # Trainable classification head
     if name.startswith("classifier"):
        param.requires_grad = True

      # Freeze everything else
     else:
        param.requires_grad = False


> **📖 O que está acontecendo aqui?**
>
> Verificamos se o freeze foi aplicado corretamente: `True` = treinável 🔥, `False` = congelado ❄️.
> Apenas `classifier.weight` e `classifier.bias` devem aparecer como `True`.
> Isso representa menos de **0,002%** dos parâmetros totais do modelo (~1.538 de ~109 milhões).


In [ ]:
# We can check whether the model was correctly updated
for name, param in model.named_parameters():
     print(f"Parameter: {name} ----- {param.requires_grad}")


Parameter: bert.embeddings.word_embeddings.weight ----- False
Parameter: bert.embeddings.position_embeddings.weight ----- False
Parameter: bert.embeddings.token_type_embeddings.weight ----- False
Parameter: bert.embeddings.LayerNorm.weight ----- False
Parameter: bert.embeddings.LayerNorm.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: bert.encoder.layer.0.attention.output

> **📖 O que está acontecendo aqui?**
>
> Treinamos com o modelo quase totalmente congelado. O treino ficará **muito mais rápido** — o PyTorch não precisa
> calcular gradientes para os parâmetros congelados. Porém, o F1 esperado cai para **~0.63**:
> as representações genéricas do BERT (treinado na Wikipedia) não são adequadas para sentimento de reviews de filmes,
> e a tiny classification head sozinha não consegue compensar isso.


In [ ]:
from transformers import TrainingArguments, Trainer

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

{'loss': '0.6947', 'grad_norm': '3.803', 'learning_rate': '1.311e-06', 'epoch': '0.9363'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '33.24', 'train_samples_per_second': '256.6', 'train_steps_per_second': '16.07', 'train_loss': '0.6949', 'epoch': '1'}


TrainOutput(global_step=534, training_loss=0.6948552792438407, metrics={'train_runtime': 33.2361, 'train_samples_per_second': 256.649, 'train_steps_per_second': 16.067, 'train_loss': 0.6948552792438407, 'epoch': 1.0})

In [ ]:
trainer.evaluate()


{'eval_loss': '0.6825', 'eval_f1': '0.64', 'eval_runtime': '3.808', 'eval_samples_per_second': '279.9', 'eval_steps_per_second': '17.59', 'epoch': '1'}


{'eval_loss': 0.6825135946273804,
 'eval_f1': 0.64,
 'eval_runtime': 3.8082,
 'eval_samples_per_second': 279.92,
 'eval_steps_per_second': 17.593,
 'epoch': 1.0}

---
## 📝 Questão 2 — Freeze Total

**Verdadeiro ou Falso — marque (V) ou (F) e justifique as falsas:**

( ) 1. Setar `requires_grad = False` em um parâmetro impede que ele seja atualizado durante o backpropagation.

( ) 2. No freeze total, a classification head também é congelada.

( ) 3. O treino com freeze total é mais rápido do que o fine-tuning completo.

( ) 4. O F1 Score com freeze total (~0.63) é maior do que com fine-tuning completo (~0.85).

( ) 5. `name.startswith("classifier")` identifica corretamente tanto `classifier.weight` quanto `classifier.bias`.

( ) 6. Com freeze total, menos de 1% dos parâmetros do modelo são atualizados durante o treino.


**Questão dissertativa:**

Por que o F1 Score cai tanto (~0.63) quando congelamos todos os encoders e treinamos apenas a classification head?
Explique o papel das representações internas do BERT nesse resultado.


### Freeze blocks 1-5


> **📖 O que está acontecendo aqui?**
>
> Antes de aplicar o freeze parcial, usamos `enumerate` para ver o **índice numérico** de cada parâmetro.
> Isso é necessário porque a próxima estratégia usa o índice (e não o nome) para decidir o que congelar.
> O Encoder Block 10 começa no índice 165 — tudo com `index < 165` corresponde aos encoders 0–9 + embeddings.


In [17]:
# We can check whether the model was correctly updated
for index, (name, param) in enumerate(model.named_parameters()):
     print(f"Parameter: {index}{name} ----- {param.requires_grad}")


Parameter: 0bert.embeddings.word_embeddings.weight ----- False
Parameter: 1bert.embeddings.position_embeddings.weight ----- False
Parameter: 2bert.embeddings.token_type_embeddings.weight ----- False
Parameter: 3bert.embeddings.LayerNorm.weight ----- False
Parameter: 4bert.embeddings.LayerNorm.bias ----- False
Parameter: 5bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: 6bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: 7bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: 8bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: 9bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: 10bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: 11bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: 12bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: 13bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: 14bert.encoder.laye

> **📖 O que está acontecendo aqui?**
>
> Aplicamos o **freeze parcial**: congelamos apenas os encoders 0 a 9 (`index < 165`) e liberamos os encoders 10 e 11
> mais a classification head para treinar.
>
> A lógica: encoders iniciais capturam padrões **gerais** (gramática, sintaxe) — já funcionam bem para qualquer tarefa.
> Encoders finais capturam representações **semânticas de alto nível** — esses se beneficiam de adaptação para a tarefa.
>
> Resultado esperado: **F1 ~0.80** — 94% da performance máxima com muito menos custo computacional.


In [18]:
# Load model
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Encoder block 10 starts at index 165 and
# we freeze everything before that block
for index, (name, param) in enumerate(model.named_parameters()):
    if index < 165:
        param.requires_grad = False

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.4761', 'grad_norm': '3.511', 'learning_rate': '1.311e-06', 'epoch': '0.9363'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '65.09', 'train_samples_per_second': '131', 'train_steps_per_second': '8.204', 'train_loss': '0.4725', 'epoch': '1'}
{'eval_loss': '0.4101', 'eval_f1': '0.8073', 'eval_runtime': '4.799', 'eval_samples_per_second': '222.1', 'eval_steps_per_second': '13.96', 'epoch': '1'}


{'eval_loss': 0.4100649952888489,
 'eval_f1': 0.8072519083969466,
 'eval_runtime': 4.7988,
 'eval_samples_per_second': 222.137,
 'eval_steps_per_second': 13.962,
 'epoch': 1.0}

---
## 📝 Questão 3 — Freeze Parcial

**Verdadeiro ou Falso — marque (V) ou (F) e justifique as falsas:**

( ) 1. No freeze parcial com `index < 165`, os encoders 10 e 11 permanecem treináveis.

( ) 2. Encoders iniciais (0–3) capturam representações semânticas mais ricas do que os encoders finais (9–11).

( ) 3. O freeze parcial resulta em um F1 Score maior do que o freeze total.

( ) 4. O valor 165 é o índice onde o Encoder Block 11 começa.

( ) 5. Usar freeze parcial em vez de fine-tuning completo sempre resulta em performance igual ou superior.

( ) 6. O freeze parcial é uma boa estratégia quando há restrições de memória ou tempo de treino.


**Questão dissertativa:**

Compare as três estratégias vistas até agora (fine-tuning completo, freeze total, freeze parcial) em termos de
**F1 Score**, **velocidade de treino** e **custo computacional**. Em qual cenário do mundo real você escolheria cada uma?


### [BONUS] Freeze blocks


> **📖 O que está acontecendo aqui?**
>
> Código comentado que treina o modelo **12 vezes**, variando quantos encoders ficam treináveis (de nenhum até todos).
> O objetivo é reproduzir o Figure 11-7 do livro: mostrar como o F1 varia progressivamente com o número de encoders livres.
>
> ⚠️ Código comentado por padrão — pode levar muito tempo. Os valores já calculados estão na célula do gráfico abaixo.


In [ ]:
# scores = []
# for index in range(12):
#     # Re-load model
#     model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)
#     tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

#     # Freeze encoder blocks 0-index
#     for name, param in model.named_parameters():
#         if "layer" in name:
#             layer_nr = int(name.split("layer")[1].split(".")[1])
#             if layer_nr <= index:
#                 param.requires_grad = False
#         else:
#             param.requires_grad = True

#     # Train
#     trainer = Trainer(
#       model=model,
#       args=training_args,
#       train_dataset=tokenized_train,
#       eval_dataset=tokenized_test,
#       tokenizer=tokenizer,
#       data_collator=data_collator,
#       compute_metrics=compute_metrics,
#     )
#     trainer.train()

#     # Evaluate
#     score = trainer.evaluate()["eval_f1"]
#     scores.append(score)


In [ ]:
# scores


> **📖 O que está acontecendo aqui?**
>
> Gráfico que plota o efeito de liberar encoders progressivamente no F1 Score.
> A linha tracejada vertical mostra o ponto onde a performance começa a **estabilizar** (~4–5 encoders treináveis).
> Os valores `y` estão invertidos (`::-1`) porque a lista original vai do mais congelado para o menos congelado.
>
> 💡 **Insight:** liberar apenas 4–5 encoders já garante ~98% da performance máxima — a curva sobe rápido e estabiliza cedo.


In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Create Figure
# plt.figure(figsize=(8,4))

# # Prepare Data
# x = [f"0-{index}" for index in range(12)]
# x[0] = "None"
# x[-1] = "All"
# y = [
#     0.8541862652869239,
#     0.8525519848771267,
#     0.8514664143803217,
#     0.8506616257088847,
#     0.8398104265402844,
#     0.8391345249294448,
#     0.8377358490566037,
#     0.8433962264150944,
#     0.8258801141769743,
#     0.816247582205029,
#     0.7917485265225934,
#     0.7019400352733686
# ][::-1]

# # Stylize Figure
# plt.grid(color='#ECEFF1')
# plt.axvline(x=4, color="#EC407A", linestyle="--")
# plt.title("Effect of Frozen Encoder Blocks on Training Performance")
# plt.ylabel("F1-score")
# plt.xlabel("Trainable encoder blocks")

# # Plot Data
# plt.plot(x, y, color="black")

# # Additional Annotation
# plt.annotate(
#     'Performance stabilizing',
#     xy=(4, y[4]),
#     xytext=(4.5, y[4]-.05),
#     arrowprops=dict(
#         arrowstyle="-|>",
#         connectionstyle="arc3",
#         color="#00ACC1")
# )
# plt.savefig("multiple_frozen_blocks.png", dpi=300, bbox_inches='tight')


---
## 📝 Questão 4 — [BONUS] Gráfico de Freeze Progressivo

**Verdadeiro ou Falso — marque (V) ou (F) e justifique as falsas:**

( ) 1. O gráfico mostra que liberar mais encoders para treino sempre aumenta o F1 Score de forma linear.

( ) 2. Com apenas 4–5 encoders treináveis, o modelo já atinge desempenho próximo ao do fine-tuning completo.

( ) 3. O loop `for index in range(12)` treina 12 modelos diferentes, um para cada configuração de freeze.

( ) 4. Os valores `y` são revertidos com `[::-1]` porque a lista original vai do modelo com mais encoders congelados para o com menos.

( ) 5. A linha tracejada vertical no gráfico indica o ponto onde o F1 Score começa a cair.


**Questão dissertativa:**

Olhando os valores pré-computados no array `y` do código (antes de `[::-1]`), qual é o F1 quando nenhum encoder é treinável?
E qual é o F1 quando todos são treináveis? Calcule a diferença absoluta e explique o que ela representa.


## Few-shot Classification


> **📖 O que é Few-Shot Classification?**
>
> Na seção anterior usamos ~8.500 exemplos de treino. Mas e quando temos **pouquíssimos dados rotulados**?
> **Few-shot** significa treinar com apenas alguns exemplos por classe — aqui apenas **16 por classe** (32 no total).
>
> O framework **SetFit** resolve isso em 3 etapas:
> 1. Gera **pares** de sentenças similares (positivo) e dissimilares (negativo) a partir dos poucos exemplos
> 2. **Fine-tuna** um modelo de embeddings (SentenceTransformers) via aprendizado contrastivo
> 3. Treina um **classificador** (regressão logística por padrão) sobre os embeddings gerados


> **📖 O que está acontecendo aqui?**
>
> `sample_dataset` seleciona aleatoriamente 16 exemplos por classe do dataset de treino.
> Com 2 classes (positivo/negativo), temos apenas **32 exemplos** para treinar — comparado aos 8.500 antes!


In [ ]:
from setfit import sample_dataset

# We simulate a few-shot setting by sampling 16 examples per class
sampled_train_data = sample_dataset(tomatoes["train"], num_samples=16)

ImportError: cannot import name 'default_logdir' from 'transformers.training_args' (/usr/local/lib/python3.12/dist-packages/transformers/training_args.py)

> **📖 O que está acontecendo aqui?**
>
> Carregamos o `all-mpnet-base-v2`, um dos modelos com melhor desempenho no benchmark MTEB de embeddings.
> O `SetFitModel` encapsula tanto o modelo de embeddings (SentenceTransformers) quanto o classificador final.


In [ ]:
from setfit import SetFitModel

# Load a pre-trained SentenceTransformer model
model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")


ImportError: cannot import name 'default_logdir' from 'transformers.training_args' (/usr/local/lib/python3.12/dist-packages/transformers/training_args.py)

> **📖 O que está acontecendo aqui?**
>
> Configuramos o `SetFitTrainer`:
> - `num_epochs=3` → épocas para o **aprendizado contrastivo** (fine-tuning dos embeddings)
> - `num_iterations=20` → pares de sentenças gerados por exemplo rotulado
>
> Com 32 exemplos e `num_iterations=20`, serão gerados `20 × 32 × 2 = 1.280 pares` automaticamente!


In [ ]:
from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer

# Define training arguments
args = SetFitTrainingArguments(
    num_epochs=3, # The number of epochs to use for contrastive learning
    num_iterations=20  # The number of text pairs to generate
)
args.eval_strategy = args.evaluation_strategy

# Create trainer
trainer = SetFitTrainer(
    model=model,
    args=args,
    train_dataset=sampled_train_data,
    eval_dataset=test_data,
    metric="f1"
)


In [ ]:
# from setfit import SetFitTrainer

# # Create trainer
# trainer = SetFitTrainer(
#     model=model,
#     train_dataset=sampled_train_data,
#     eval_dataset=test_data,
#     metric="f1",
#     num_epochs=3, # The number of epochs to use for contrastive learning
# )


> **📖 O que está acontecendo aqui?**
>
> O SetFit executa automaticamente as 3 etapas: geração de pares → fine-tuning dos embeddings → treino do classificador.
> No output você verá: `Num unique pairs = 1280` — confirmando a geração automática dos pares de treino.


In [ ]:
# Training loop
trainer.train()


> **📖 O que está acontecendo aqui?**
>
> Avaliamos no conjunto completo de teste. O F1 esperado é **~0.84** — com apenas **32 exemplos de treino**!
> Isso demonstra a eficiência do SetFit: ele extrai máximo valor dos poucos dados disponíveis.


In [ ]:
# Evaluate the model on our test data
trainer.evaluate()


> **📖 O que está acontecendo aqui?**
>
> `model.model_head` exibe o classificador final — por padrão uma **regressão logística** do scikit-learn.
> É possível substituí-lo por qualquer classificador scikit-learn ou por uma rede neural diferenciável
> usando `use_differentiable_head=True` no `SetFitModel.from_pretrained()`.


In [ ]:
model.model_head


---
## 📝 Questão 5 — Few-Shot Classification com SetFit

**Verdadeiro ou Falso — marque (V) ou (F) e justifique as falsas:**

( ) 1. O SetFit utiliza apenas 16 exemplos por classe para treinar o modelo.

( ) 2. O aprendizado contrastivo no SetFit usa pares de sentenças similares (positivos) e dissimilares (negativos).

( ) 3. Com `num_iterations=20` e 32 exemplos, o SetFit gera exatamente 320 pares de sentenças.

( ) 4. O classificador final do SetFit é, por padrão, uma rede neural profunda.

( ) 5. O F1 Score do SetFit com 32 exemplos é comparável ao do fine-tuning completo com 8.500 exemplos.

( ) 6. O SetFit é uma boa escolha quando há abundância de dados rotulados disponíveis.


**Questão dissertativa:**

Explique as 3 etapas do algoritmo SetFit. Por que gerar pares de sentenças a partir de dados de classificação
(e não de pares já rotulados como similares/dissimilares) é uma estratégia inteligente?


## MLM


> **📖 O que é Continued Pretraining com MLM?**
>
> O fluxo padrão é: *pré-treino → fine-tuning*. Aqui inserimos um passo intermediário:
> continuar o pré-treino do BERT em dados do nosso domínio usando **Masked Language Modeling (MLM)**.
>
> **Como o MLM funciona:** mascaramos aleatoriamente 15% dos tokens de cada frase com `[MASK]`.
> O modelo aprende a **prever os tokens mascarados** a partir do contexto — sem nenhum label externo.
> Isso adapta as representações internas do BERT para o vocabulário do novo domínio.
>
> 💡 **Analogia:** É como fazer o BERT "ler" centenas de reviews de filmes antes de aprender a classificá-los.
> Ele aprende que "cinematografia", "roteiro" e "atuação" são palavras centrais nesse contexto.


> **📖 O que está acontecendo aqui?**
>
> Usamos `AutoModelForMaskedLM` — versão do BERT para a tarefa de prever tokens mascarados.
> É diferente do `AutoModelForSequenceClassification` usado antes: não há classification head aqui,
> apenas o BERT base com uma camada de projeção para o vocabulário.


In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Load model for Masked Language Modeling (MLM)
model = AutoModelForMaskedLM.from_pretrained("bert-base-cased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


> **📖 O que está acontecendo aqui?**
>
> Tokenizamos os dados para o MLM. Diferente da classificação, **removemos a coluna `label`** —
> o MLM é não supervisionado: a "resposta certa" é o próprio token mascarado, não uma classe externa.
> Usamos os mesmos reviews do Rotten Tomatoes como corpus de continued pretraining.


In [ ]:
def preprocess_function(examples):
   return tokenizer(examples["text"], truncation=True)

# Tokenize data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_train = tokenized_train.remove_columns("label")
tokenized_test = test_data.map(preprocess_function, batched=True)
tokenized_test = tokenized_test.remove_columns("label")


> **📖 O que está acontecendo aqui?**
>
> `DataCollatorForLanguageModeling` aplica o mascaramento automaticamente durante o treino:
> - `mlm=True` → ativa o modo de masked language modeling
> - `mlm_probability=0.15` → mascara 15% dos tokens aleatoriamente em cada batch
>
> Em cada epoch, tokens **diferentes** são mascarados, aumentando a diversidade dos exemplos.


In [ ]:
from transformers import DataCollatorForLanguageModeling

# Masking Tokens
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)


> **📖 Alternativa comentada — Whole-Word Masking:**
>
> Em vez de mascarar tokens individuais (podendo pegar apenas parte de uma palavra),
> o `DataCollatorForWholeWordMask` mascara a **palavra inteira**.
> Ex: `"cinemat ##ografia"` → ambos os tokens são mascarados juntos.
> É mais desafiador para o modelo e tende a produzir representações melhores, mas converge mais devagar.


In [ ]:
# from transformers import DataCollatorForWholeWordMask

# # Masking Whole Words
# data_collator = DataCollatorForWholeWordMask(
#     tokenizer=tokenizer,
#     mlm=True,
#     mlm_probability=0.15
# )


> **📖 O que está acontecendo aqui?**
>
> Configuramos o `Trainer` para o pré-treino continuado. Diferenças em relação à classificação:
> - `num_train_epochs=10` → pré-treino precisa de mais epochs para adaptar bem as representações
> - **Sem `compute_metrics`** — o MLM usa a loss de predição como métrica interna
>
> Após o treino, salvamos o tokenizador (não muda) e o modelo atualizado na pasta `"mlm/"`,
> que poderá ser usado como ponto de partida para um fine-tuning de classificação posterior.


In [ ]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=10,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)


# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator
)


In [ ]:
# Save pre-trained tokenizer
tokenizer.save_pretrained("mlm")

# Train model
trainer.train()

# Save updated model
model.save_pretrained("mlm")


> **📖 O que está acontecendo aqui?**
>
> Testamos o modelo **original** na tarefa de preencher `[MASK]` em `"What a horrible [MASK]!"`.
> Esperamos respostas **genéricas** como "idea", "dream", "day" — palavras comuns na Wikipedia.


In [ ]:
from transformers import pipeline

# Load and create predictions
mask_filler = pipeline("fill-mask", model="bert-base-cased")
preds = mask_filler("What a horrible [MASK]!")

# Print results
for pred in preds:
    print(f">>> {pred['sequence']}")


> **📖 O que está acontecendo aqui?**
>
> Agora testamos o modelo **após o continued pretraining** nos reviews do Rotten Tomatoes.
> As respostas devem ser relacionadas ao domínio de **filmes**: "movie", "film", "comedy".
> Isso confirma que o MLM adaptou as representações internas do modelo para o novo domínio.


In [ ]:
# Load and create predictions
mask_filler = pipeline("fill-mask", model="mlm")
preds = mask_filler("What a horrible [MASK]!")

# Print results
for pred in preds:
    print(f">>> {pred['sequence']}")


---
## 📝 Questão 6 — Continued Pretraining com MLM

**Verdadeiro ou Falso — marque (V) ou (F) e justifique as falsas:**

( ) 1. O `AutoModelForMaskedLM` é a mesma arquitetura do `AutoModelForSequenceClassification`, apenas com parâmetros diferentes.

( ) 2. A coluna `label` é removida do dataset no MLM porque essa tarefa é não supervisionada.

( ) 3. O `DataCollatorForLanguageModeling` com `mlm_probability=0.15` mascara exatamente os mesmos tokens em todas as epochs.

( ) 4. O modelo após continued pretraining nos reviews de filmes tende a prever palavras relacionadas a filmes no lugar de `[MASK]`.

( ) 5. O continued pretraining com MLM utiliza `num_train_epochs=10`, mais do que no fine-tuning de classificação com `num_train_epochs=1`.

( ) 6. Após o continued pretraining, o modelo salvo em `"mlm/"` pode ser carregado diretamente com `AutoModelForSequenceClassification` para fine-tuning de classificação.


**Questão dissertativa:**

Explique a diferença entre **token masking** e **whole-word masking**. Para a tarefa de classificação de sentimento
de reviews de filmes, qual das duas estratégias você esperaria que produzisse melhores resultados e por quê?


## Named Entity Recognition

Here are a number of interesting datasets you can also explore for NER:
* tner/mit_movie_trivia
* tner/mit_restaurant
* wnut_17
* conll2003


> **📖 O que é Named Entity Recognition (NER)?**
>
> NER é uma tarefa de **classificação no nível do token**: em vez de classificar um documento inteiro,
> classificamos cada palavra individualmente para identificar entidades como:
> - **PER** → pessoas ("Maarten", "Barack Obama")
> - **ORG** → organizações ("Google", "Rangers")
> - **LOC** → localizações ("Netherlands", "New York")
> - **MISC** → miscelânea ("World Cup")
> - **O** → não é entidade
>
> O sistema **BIO** (Beginning-Inside-Outside) é usado: `B-PER` inicia uma entidade pessoa,
> `I-PER` continua a mesma entidade. Assim `"Dean Palmer"` → `Dean: B-PER`, `Palmer: I-PER`.


In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import numpy as np


> **📖 O que está acontecendo aqui?**
>
> Carregamos o **CoNLL-2003**, benchmark padrão para NER em inglês com ~14.000 frases de notícias anotadas.
> Cada palavra já vem com sua entidade correspondente — os dados estão pré-tokenizados por palavras (não subwords).


In [ ]:
# The CoNLL-2003 dataset for NER
dataset = load_dataset("conll2003", trust_remote_code=True)


> **📖 O que está acontecendo aqui?**
>
> Inspecionamos um exemplo. O campo `tokens` contém as palavras da frase e `ner_tags` os IDs de entidade correspondentes.
> Ex: `ner_tags = [1, 2, 0, ...]` significa: primeira palavra = `B-PER`, segunda = `I-PER`, terceira = `O` (sem entidade).


In [ ]:
example = dataset["train"][848]
example


> **📖 O que está acontecendo aqui?**
>
> Definimos os mapeamentos entre labels textuais e IDs numéricos.
> O prefixo **B** (Beginning) indica o início de uma entidade; **I** (Inside) indica continuação da mesma entidade.
> Isso evita ambiguidade: dois nomes seguidos (`"Dean Palmer"` e `"John Smith"`) seriam `B-PER I-PER B-PER I-PER`.


In [ ]:
label2id = {
    'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4,
    'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8
}
id2label = {index: label for label, index in label2id.items()}
label2id


> **📖 O que está acontecendo aqui?**
>
> Para NER usamos `AutoModelForTokenClassification` — ele faz uma **predição por token** (não por documento).
> Passamos `id2label` e `label2id` para que o modelo conheça os nomes das 9 classes de entidade.


In [ ]:
from transformers import AutoModelForTokenClassification

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Load model
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)


> **📖 O que está acontecendo aqui?**
>
> O BERT usa tokenização por **subwords** — pode dividir palavras em múltiplos tokens.
> Ex: `"homer"` → `"home"` + `"##r"`. Isso cria um problema: os labels estão no nível de **palavra**,
> mas o modelo trabalha com **tokens**. A próxima célula (`align_labels`) resolve esse desalinhamento.


In [ ]:
# Split individual tokens into sub-tokens
token_ids = tokenizer(example["tokens"], is_split_into_words=True)["input_ids"]
sub_tokens = tokenizer.convert_ids_to_tokens(token_ids)
sub_tokens


> **📖 O que está acontecendo aqui?**
>
> A função `align_labels` resolve o desalinhamento palavra → subtoken com as seguintes regras:
> - **Primeiro token de uma palavra** → recebe o label original (ex: `B-PER`)
> - **Tokens subsequentes da mesma palavra** → `B-XXX` vira `I-XXX` (continuação)
> - **Tokens especiais** `[CLS]` e `[SEP]` → recebem `-100` (ignorados pela loss)
>
> Ex: `"Maarten"` (B-PER) tokenizado como `"Ma"`, `"##arte"`, `"##n"` → labels: `B-PER`, `I-PER`, `I-PER`


In [ ]:
def align_labels(examples):
    token_ids = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = examples["ner_tags"]

    updated_labels = []
    for index, label in enumerate(labels):

        # Map tokens to their respective word
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:

            # The start of a new word
            if word_idx != previous_word_idx:

                previous_word_idx = word_idx
                updated_label = -100 if word_idx is None else label[word_idx]
                label_ids.append(updated_label)

            # Special token is -100
            elif word_idx is None:
                label_ids.append(-100)

            # If the label is B-XXX we change it to I-XXX
            else:
                updated_label = label[word_idx]
                if updated_label % 2 == 1:
                    updated_label += 1
                label_ids.append(updated_label)

        updated_labels.append(label_ids)

    token_ids["labels"] = updated_labels
    return token_ids

tokenized = dataset.map(align_labels, batched=True)


> **📖 O que está acontecendo aqui?**
>
> Comparamos os labels originais (nível de palavra) com os labels alinhados (nível de subtoken).
> Note os `-100` adicionados para `[CLS]` e `[SEP]` — eles são ignorados pela função de perda durante o treino.
> O tamanho da lista atualizada é maior que o original porque palavras podem gerar múltiplos subtokens.


In [ ]:
# Difference between original and updated labels
print(f"Original: {example['ner_tags']}")
print(f"Updated: {tokenized['train'][848]['labels']}")


> **📖 O que está acontecendo aqui?**
>
> Para NER, a métrica é calculada no nível da entidade (não do token individual) usando `seqeval`.
> - Ignoramos tokens com label `-100` (tokens especiais e subtokens não-iniciais)
> - `id2label[token_prediction]` converte o ID numérico de volta para o label textual (`"B-PER"` etc.)
> - `seqeval` calcula precisão, recall e F1 **por tipo de entidade** (PER, ORG, LOC, MISC)


In [ ]:
import evaluate

# Load sequential evaluation
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    # Create predictions
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = []
    true_labels = []

    # Document-level iteration
    for prediction, label in zip(predictions, labels):

      # token-level iteration
      for token_prediction, token_label in zip(prediction, label):

        # We ignore special tokens
        if token_label != -100:
          true_predictions.append([id2label[token_prediction]])
          true_labels.append([id2label[token_label]])

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {"f1": results["overall_f1"]}


> **📖 O que está acontecendo aqui?**
>
> Usamos `DataCollatorForTokenClassification` — diferente do `DataCollatorWithPadding` da classificação,
> este collator também faz **padding nas listas de labels** (preenchendo com `-100`),
> garantindo que tokens e labels tenham o mesmo tamanho dentro de cada batch.


In [ ]:
from transformers import DataCollatorForTokenClassification

# Token-classification Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


> **📖 O que está acontecendo aqui?**
>
> Treinamos o modelo de NER. O processo é idêntico ao da classificação de sequências, exceto:
> - O dataset é o CoNLL-2003 tokenizado e com labels alinhados
> - O modelo faz predições por token (não por documento)
> - As métricas usam `seqeval` no lugar do F1 simples


In [ ]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()


> **📖 O que está acontecendo aqui?**
>
> Avaliamos o modelo de NER. O `seqeval` retorna o F1 geral (`overall_f1`) e também por tipo de entidade.


In [ ]:
# Evaluate the model on our test data
trainer.evaluate()


> **📖 O que está acontecendo aqui?**
>
> Salvamos o modelo fine-tunado e rodamos uma **inferência de exemplo** para verificar se funciona.
> A frase `"My name is Maarten."` deve ter `"Maarten"` identificado como `B-PER` / `I-PER`.
> Note que o output retorna subtokens separados: `"Ma"` (B-PER), `"##arte"` (I-PER), `"##n"` (I-PER).


In [ ]:
from transformers import pipeline

# Save our fine-tuned model
trainer.save_model("ner_model")

# Run inference on the fine-tuned model
token_classifier = pipeline(
    "token-classification",
    model="ner_model",
)
token_classifier("My name is Maarten.")


---
## 📝 Questão 7 — Named Entity Recognition

**Verdadeiro ou Falso — marque (V) ou (F) e justifique as falsas:**

( ) 1. O `AutoModelForTokenClassification` classifica o documento inteiro, assim como o `AutoModelForSequenceClassification`.

( ) 2. O sistema BIO usa "B" para marcar o início de uma entidade e "I" para marcar a continuação da mesma entidade.

( ) 3. Na função `align_labels`, tokens especiais como `[CLS]` e `[SEP]` recebem o label `-100` para serem ignorados pela loss.

( ) 4. O `DataCollatorForTokenClassification` é idêntico ao `DataCollatorWithPadding`, apenas com outro nome.

( ) 5. A palavra "Maarten" tokenizada como `"Ma"`, `"##arte"`, `"##n"` resulta nos labels `B-PER`, `I-PER`, `I-PER`.

( ) 6. O `seqeval` calcula o F1 Score apenas para a classe `PER`, ignorando as demais entidades.


**Questão dissertativa:**

Por que o problema de desalinhamento entre palavras e subtokens existe no NER mas não existe na classificação de sentimento?
Explique como a função `align_labels` resolve esse problema e qual seria o impacto de não fazer esse alinhamento.


In [ ]:
import transformers
print(transformers.__version__)

4.37.2
